# TechJam Phase 3: BGE vs OpenAI embeddings

This notebook runs the unchanged controlled retrieval comparison on a Colab GPU. Caches and results persist to Google Drive. **Do not run the full benchmark until the catalog count and batch estimate cell looks correct.**

## 1. Enable a GPU first

In Colab choose **Runtime > Change runtime type > T4 GPU** (or better), then run this cell.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU detected. Change the Colab runtime to T4 GPU or better.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

## 2. Mount Drive and extract the prepared bundle

The default ZIP location is `MyDrive/techjam26_phase3/phase3_colab_bundle.zip`. If it is absent, this cell offers a direct browser upload.

In [ ]:
from google.colab import drive, files
from pathlib import Path, PurePosixPath
import shutil
from zipfile import ZipFile

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/techjam26_phase3')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
bundle = DRIVE_ROOT / 'phase3_colab_bundle.zip'
if not bundle.exists():
    print('Bundle not found in Drive; choose phase3_colab_bundle.zip now.')
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    # files.upload() writes to the notebook's current directory, which is
    # not guaranteed to be /content after Drive or other notebook cells.
    bundle = (Path.cwd() / uploaded_name).resolve()

assert bundle.exists(), f'Bundle path does not exist: {bundle}'
EXTRACT_ROOT = Path('/content/phase3_extracted')
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)
target_suffix = 'nickolas/shopping_agent/compare_embeddings.py'
script_path = None
with ZipFile(bundle) as archive:
    for member in archive.infolist():
        # Windows Compress-Archive can preserve backslashes. Normalize every
        # member manually so Linux receives real directory components.
        normalized = member.filename.replace('\\', '/').lstrip('/')
        relative = PurePosixPath(normalized)
        assert '..' not in relative.parts, f'Unsafe ZIP member: {member.filename}'
        destination = EXTRACT_ROOT.joinpath(*relative.parts)
        if member.is_dir() or normalized.endswith('/'):
            destination.mkdir(parents=True, exist_ok=True)
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, destination.open('wb') as output:
            shutil.copyfileobj(source, output)
        if normalized.endswith(target_suffix):
            script_path = destination

assert script_path is not None and script_path.exists(), 'Comparison script was not extracted'
WORKSPACE = script_path.parents[2]
assert (WORKSPACE / 'techjam-conversational-search/data/catalog.jsonl').exists()
print('Bundle:', bundle)
print('Workspace:', WORKSPACE)

## 3. Install the exact notebook dependencies

In [ ]:
import subprocess, sys
requirements = WORKSPACE / 'nickolas/colab/requirements-phase3-colab.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
print('Dependencies installed.')

## 4. Load the OpenAI key from Colab Secrets

Create a Colab Secret named `OPENAI_API_KEY` and enable notebook access. The value is not printed or written to disk.

In [ ]:
from google.colab import userdata
import os

openai_key = userdata.get('OPENAI_API_KEY')
assert openai_key, 'Add OPENAI_API_KEY to Colab Secrets and enable notebook access.'
os.environ['OPENAI_API_KEY'] = openai_key
os.environ['BGE_EMBEDDING_BATCH_SIZE'] = '64'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('OpenAI key loaded:', bool(os.environ.get('OPENAI_API_KEY')))
print('BGE batch size:', os.environ['BGE_EMBEDDING_BATCH_SIZE'])

## 5. Run all mock-only tests

This must not download BGE or call OpenAI.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'nickolas/shopping_agent/tests', '-v'], cwd=WORKSPACE, check=True)

## 6. Verify scope before the expensive call

In [ ]:
import math
catalog = WORKSPACE / 'techjam-conversational-search/data/catalog.jsonl'
catalog_count = sum(1 for line in catalog.open(encoding='utf-8') if line.strip())
openai_batch_size = 1000
print('Catalog product texts:', catalog_count)
print('Expected OpenAI catalog request batches:', math.ceil(catalog_count / openai_batch_size))
assert catalog_count == 50_000

OUTPUT_ROOT = DRIVE_ROOT / 'outputs'
CACHE_DIR = OUTPUT_ROOT / 'embedding_cache'
RESULTS_DIR = OUTPUT_ROOT / 'benchmark_results'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Persistent cache:', CACHE_DIR)
print('Persistent results:', RESULTS_DIR)

## 7. Small hosted smoke test

This sends one short embedding request and does not embed the catalog.

In [ ]:
compare_script = WORKSPACE / 'nickolas/shopping_agent/compare_embeddings.py'
subprocess.run([
    sys.executable, str(compare_script),
    '--results-dir', str(RESULTS_DIR),
    '--cache-dir', str(CACHE_DIR),
    'smoke-openai',
], cwd=WORKSPACE, check=True)

## 8. Explicit full controlled retrieval benchmark

This is the expensive cell. It builds missing validated caches for vanilla BGE and OpenAI, then scores the exact same 200 retrieval queries. BGE uses the GPU. OpenAI uses 50 batched catalog requests plus query requests. Re-running loads valid caches instead of rebuilding them.

In [ ]:
subprocess.run([
    sys.executable, str(compare_script),
    '--results-dir', str(RESULTS_DIR),
    '--cache-dir', str(CACHE_DIR),
    'retrieval',
    '--samples', '200',
    '--backends', 'bge', 'openai',
    '--allow-openai-catalog-build',
], cwd=WORKSPACE, check=True)

## 9. Display the primary comparison

In [ ]:
from IPython.display import Markdown, display
summary_path = RESULTS_DIR / 'comparison_summary.md'
assert summary_path.exists(), 'Benchmark summary was not produced.'
display(Markdown(summary_path.read_text(encoding='utf-8')))

## 10. Optional stochastic end-to-end evaluator

Disabled by default. This can make many additional hosted chat calls. It uses the same shared evaluator for both variants, but the shopper is stochastic, so controlled retrieval remains primary.

In [ ]:
RUN_END_TO_END = False
if RUN_END_TO_END:
    subprocess.run([
        sys.executable, str(compare_script),
        '--results-dir', str(RESULTS_DIR),
        '--cache-dir', str(CACHE_DIR),
        'end-to-end',
        '--samples', '200',
        '--backends', 'bge', 'openai',
        '--repeats', '1',
    ], cwd=WORKSPACE, check=True)
else:
    print('Skipped. Set RUN_END_TO_END=True only when you intentionally want the stochastic hosted evaluation.')

## 11. List persistent artifacts

The NPZ files can be copied directly into local `nickolas/shopping_agent/embedding_cache/`.

In [ ]:
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file():
        print(path.relative_to(OUTPUT_ROOT), f'{path.stat().st_size / (1024**2):.2f} MB')